
<h1 style="text-align: center;">CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro de Atividade Prática</h1>
<br>
<br>

Nome: ______________________________________________________________________________________      

Turma: ______________


**Componente:** Inteligência Artificial 
<br>
**Unidade Curricular:** Redes Neurais
<br>
**Tema da Semana:** Transfer learning
<br>


# Aula 2: Transfer learning

## Tarefa
- Execute o código abaixo.

- Se as bibliotecas não tiverem instaladas, instale-as.

- Acompanhe as instruções.

- Responda as perguntas e conclua a atividade


### Instalação das dependências

In [1]:
#!pip install --upgrade pip setuptools wheel

In [2]:
#!pip install tensorflow==2.2.0

In [3]:
#!pip install  Pillow  matplotlib

In [4]:
#!conda install anaconda::tensorflow

### Descompressão das imagens

In [5]:
!unzip -o -qq train.zip

In [6]:
!unzip -o -qq test.zip

### Imports de dependências

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D
from tensorflow.keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet import preprocess_input
import matplotlib.pyplot as plt


In [ ]:
: Por exemplo, em visão computacional, pode-se usar modelos como ResNet ou VGG, e em NLP, usar BERT ou GPT.​

​

Remover ou ajustar as últimas camadas: Essas camadas são específicas para a tarefa original (por exemplo, 1000 classes do ImageNet), e precisam ser ajustadas ou substituídas para se adequar à nova tarefa (por exemplo, classificação binária ou classificação multiclasse).​

​

Congelar ou ajustar as camadas iniciais do modelo: As primeiras camadas geralmente capturam características universais (como bordas e texturas no caso de imagens, ou relações de palavras no caso de texto). Essas camadas podem ser congeladas para acelerar o treinamento, ou ajustadas para refinar o aprendizado para a nova tarefa.​

​

Treinar o modelo com o novo conjunto de dados: Usar o novo conjunto de dados para ajustar o modelo, com foco nas camadas superiores. Isso pode ser feito de forma rápida, pois o modelo já aprendeu características relevantes de outra tarefa

### Configurações

In [8]:
SEED = 42


def set_seed(seed=0):
  np.random.seed(seed) 
  tf.random.set_seed(seed) 
  random.seed(seed)
  os.environ['TF_DETERMINISTIC_OPS'] = "1"
  os.environ['TF_CUDNN_DETERMINISM'] = "1"
  os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# Caminho das pastas com as imagens
train_data_dir = 'pet_images_sample/train'
validation_data_dir = 'pet_images_sample/test'


# Tamanho das imagens de entrada
img_rows, img_cols = 224, 224


# Quantidade de Classes
num_classes = 2

# Tamanho das amostras de treinamento e validação
nb_train_samples = 1600
nb_validation_samples = 400

# Número de épocas de treinamento
epochs = 3

# Tamanho do batch size
batch_size = 32

### Leitura dos dados

In [9]:
# Configuração do gerador de dados para treinamento com data augmentation 
# para garantir que o modelo generaliza gerando imagens muito próximas das originais
train_datagen = ImageDataGenerator(
    rescale=1./255,                 # Normaliza os pixels para o intervalo [0, 1]
    rotation_range=45,              # Rotaciona imagens aleatoriamente até 45 graus
    width_shift_range=0.3,          # Desloca horizontalmente até 30% da largura
    height_shift_range=0.3,         # Desloca verticalmente até 30% da altura
    horizontal_flip=True,           # Espelha horizontalmente as imagens
    fill_mode='nearest'             # Preenche os espaços vazios com o pixel mais próximo
)

validation_datagen = ImageDataGenerator(rescale=1./255)


train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical'
)

Found 1600 images belonging to 2 classes.
Found 400 images belonging to 2 classes.


### Criação o modelo

In [10]:
# Carrega o modelo pré-treinado
MobileNet = MobileNet(weights='imagenet', include_top=False, input_shape=(img_rows, img_cols, 3))


# Substitui as últimas camadas para se adequar à nova tarefa
def add_model(bottom_model, num_classes):

  top_model = bottom_model.output
  top_model = GlobalAveragePooling2D() (top_model)

  top_model = Dense (1024, activation='relu') (top_model)
  top_model = Dense (1024, activation='relu') (top_model)

  top_model = Dense (512, activation='relu') (top_model)
  top_model = Dense (num_classes, activation='softmax') (top_model)

  return top_model


# Congela as camadas iniciais do model
for layer in MobileNet.layers:
    layer.trainable = False
    

FC_Head = add_model(MobileNet, num_classes)

model = Model(inputs = MobileNet.input, outputs = FC_Head)

2024-10-07 00:10:46.778375: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2024-10-07 00:10:46.778398: E tensorflow/stream_executor/cuda/cuda_driver.cc:313] failed call to cuInit: UNKNOWN ERROR (303)
2024-10-07 00:10:46.778417: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (fabiojanes-ThinkPad-E14): /proc/driver/nvidia/version does not exist
2024-10-07 00:10:46.778604: I tensorflow/core/platform/cpu_feature_guard.cc:143] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2024-10-07 00:10:46.804974: I tensorflow/core/platform/profile_utils/cpu_utils.cc:102] CPU Frequency: 2299965000 Hz
2024-10-07 00:10:46.805560: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7fc2b40018a0 initialized for platform Host (this does not g

### Perguntas e Conclusão

#### Qual método de Transfer Learning foi aplicado na CNN (Congelamento de Camadas ou Ajuste Fino (Fine-Tuning))?